# VOSR 2.0 · GIRAFFE

На GPU ещё не проверен.

Запускайте сверху вниз в отдельной сессии Kaggle с GPU и Internet. Зависимости устанавливаются в отдельное Python-окружение.

Подключите датасет через Add Input и укажите его путь в разделе 4. Структура: `<корень>/<sample_id>/degraded.png`; `hint.png` нужен только для просмотра.

[Официальный репозиторий](https://github.com/cswry/VOSR) · [Зафиксированная версия](https://github.com/cswry/VOSR/tree/516f292b99cf23c76fdc33351e86dc4f97711fe8)

## 1. Репозиторий


In [ ]:
from pathlib import Path
import datetime
import hashlib
import json
import os
import shutil
import struct
import subprocess
import sys
import uuid
import zipfile

REPO_URL = 'https://github.com/cswry/VOSR.git'
COMMIT = '516f292b99cf23c76fdc33351e86dc4f97711fe8'
WORK_ROOT = Path('/kaggle/working/giraffe_baselines') if Path('/kaggle/working').exists() else Path.cwd() / '.baseline_work'
WORK_ROOT = WORK_ROOT.resolve()
REPO_DIR = WORK_ROOT / 'external' / 'vosr'

def run(*args, cwd=REPO_DIR, env=None):
    subprocess.run([str(a) for a in args], cwd=cwd, env=env, check=True)

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    run('git', 'clone', REPO_URL, REPO_DIR, cwd=WORK_ROOT)
origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.removesuffix('.git') != REPO_URL.removesuffix('.git'):
    raise RuntimeError(f'В {REPO_DIR} находится другой репозиторий: {origin}')
if subprocess.check_output(['git', '-C', str(REPO_DIR), 'diff', 'HEAD', '--name-only'], text=True).strip():
    raise RuntimeError('В baseline-репозитории изменены отслеживаемые файлы. Сохраните изменения отдельно.')
run('git', 'fetch', 'origin', COMMIT)
run('git', 'checkout', '--detach', COMMIT)
actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual != COMMIT:
    raise RuntimeError(f'Версия репозитория не совпала: {actual}')
print(f'vosr: {actual}')


## 2. Окружение

VOSR 2.0, один шаг, увеличение ×4, Qwen Image VAE и DINOv2-L. Устанавливаются зависимости инференса: полный requirements авторов содержит конфликтующие пакеты для обучения и метрик. Для T4 включена штатная обработка фрагментами 512×512 в DiT и VAE. Полный проход DiT этой версии принимает только квадратные входы; VAE без разбиения не помещает крупные изображения в 16 ГБ памяти.


In [ ]:
# Зависимости метода устанавливаются отдельно от ядра Jupyter.
if sys.platform != 'linux' or not shutil.which('nvidia-smi'):
    raise RuntimeError('Нужна Linux-сессия с NVIDIA GPU. В Kaggle включите GPU и Internet. На Mac CUDA недоступна.')
run('nvidia-smi', cwd=WORK_ROOT)
run(sys.executable, '-m', 'pip', 'install', 'uv==0.9.7', cwd=WORK_ROOT)
ENV_DIR = WORK_ROOT / 'envs' / 'vosr'
if not (ENV_DIR / 'bin/python').exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', ENV_DIR, cwd=WORK_ROOT)
PYTHON = ENV_DIR / 'bin/python'

def run_python(source, *args, env=None):
    run(PYTHON, '-c', source, *args, env=env)

run(PYTHON, '-m', 'pip', 'install', 'pip==24.3.1', 'setuptools==75.6.0', 'wheel==0.45.1')
run(PYTHON, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121')
# Только зависимости inference: полный requirements включает несовместимый с Transformers 4.52 PyIQA.
run(PYTHON, '-m', 'pip', 'install', 'diffusers==0.35.0', 'transformers==4.52.0',
    'accelerate==1.1.0', 'fairscale==0.4.13', 'timm==1.0.11', 'einops==0.8.0',
    'numpy==1.26.4', 'opencv-python-headless==4.10.0.84', 'Pillow==11.1.0',
    'safetensors==0.5.3', 'tqdm==4.67.1', 'huggingface-hub==0.34.4', 'gdown==5.2.0')
run_python("import torch; assert torch.cuda.is_available(), 'PyTorch не видит CUDA'; print(torch.__version__, torch.cuda.get_device_name(0))")
run(PYTHON, '-m', 'pip', 'check')


## 3. Веса

Веса VOSR2, Qwen Image VAE и DINOv2 скачиваются из репозитория `CSWRY/VOSR` на Hugging Face.

Загрузка начинается при запуске ячейки. Уже скачанные файлы используются повторно.


In [ ]:
run_python(r"""from pathlib import Path
import json
from huggingface_hub import snapshot_download, model_info
assets = []


def hf_snapshot(repo, destination, patterns, revision=None):
    destination = Path(destination)
    revision = revision or model_info(repo).sha
    snapshot_download(repo_id=repo, revision=revision, local_dir=destination,
                      allow_patterns=patterns, ignore_patterns=['*.onnx*', '*.msgpack', '*.fp16.*', '*.pyc'])
    assets.append({'source': f'https://huggingface.co/{repo}', 'revision': revision,
                   'local': str(destination), 'patterns': patterns})
    return destination


hf_snapshot('CSWRY/VOSR', 'preset/ckpts', [
    'VOSR2/*', 'Qwen-Image-vae-2d/*',
    'torch_cache/checkpoints/dinov2_vitl14_pretrain.pth',
    'torch_cache/facebookresearch_dinov2_main/*', 'torch_cache/trusted_list',
], revision='f24b3061b7f350b81e1907bdcfc27f71fb4ff3f3')
Path('giraffe_assets.json').write_text(json.dumps({'assets': assets}, indent=2))
""")


## 4. Параметры и данные

Вход модели — только `degraded.png`. Каждое выполнение создаёт новую папку запуска.


In [ ]:
DATASET_ROOT = Path('/kaggle/input/datasets/markunyaa/giraffe-dataset/giraffe_dataset')
RESULTS_ROOT = WORK_ROOT / 'results'
SEED = 42
SETTINGS = {
    'upscale': 4,
    'infer_steps': 1,
    'align_method': 'wavelet',
    'tile_size': 512,
    'tile_overlap': 32,
    'vae_tile_size': 512,
}

DATASET_ROOT = DATASET_ROOT.expanduser().resolve()
RESULTS_ROOT = RESULTS_ROOT.expanduser().resolve()
sources = sorted(DATASET_ROOT.glob('*/degraded.png'))
if not sources:
    raise FileNotFoundError(f'Не найдены */degraded.png в {DATASET_ROOT}. Укажите путь из Kaggle Add Input.')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = WORK_ROOT / 'runs' / 'vosr' / RUN_ID
INPUT_DIR = RUN_DIR / 'input'
RAW_DIR = RUN_DIR / 'raw'
INPUT_DIR.mkdir(parents=True)
RAW_DIR.mkdir()
SAMPLES = {}
for src in sources:
    sample_id = src.parent.name
    target = INPUT_DIR / f'{sample_id}.png'
    shutil.copy2(src, target)
    SAMPLES[sample_id] = src
ASSETS = json.loads((REPO_DIR / 'giraffe_assets.json').read_text())
RUN_CONFIG = {'method': 'vosr', 'repo': REPO_URL, 'commit': COMMIT, 'seed': SEED,
              'settings': SETTINGS, 'weights': ASSETS, 'dataset_root': str(DATASET_ROOT),
              'input_dir': str(INPUT_DIR), 'raw_dir': str(RAW_DIR), 'run_id': RUN_ID}
CONFIG_PATH = RUN_DIR / 'run.json'
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))
INFERENCE_OK = False
EXPORT_OK = False
print(f'{len(SAMPLES)} примеров: ' + ', '.join(SAMPLES))
print(f'Результаты: {RESULTS_ROOT}')


## 5. Инференс

Для повторного запуска сначала выполните ячейку параметров.


In [ ]:
INFERENCE_OK = False
EXPORT_OK = False
if any(RAW_DIR.iterdir()):
    raise RuntimeError('Для повторного инференса заново выполните ячейку параметров: она создаст новую папку запуска.')

COMMAND = [str(PYTHON), 'inference_vosr_onestep.py', '-c', 'preset/ckpts/VOSR2',
    '-i', str(INPUT_DIR), '-o', str(RAW_DIR), '--seed', str(SEED)]
for key, value in SETTINGS.items():
    COMMAND += ['--' + key, str(value)]
run(*COMMAND)
PREDICTIONS = {sid: RAW_DIR / f'{sid}.png' for sid in SAMPLES}
# Официальный скрипт перехватывает ошибки отдельных изображений и возвращает код 0.
missing = [sid for sid, path in PREDICTIONS.items() if not path.is_file()]
if missing:
    raise RuntimeError('VOSR не обработал: ' + ', '.join(missing) + '. Причина указана в логе выше.')
INFERENCE_OK = True


## 6. Сохранение

Результаты: `results/<sample_id>/vosr.png`. Рядом сохраняются параметры, версии весов, seed и размеры изображений. Архив содержит результаты и сведения о запуске. Повторный экспорт заменяет PNG и JSON этого метода; архивы сохраняются отдельно.


In [ ]:
EXPORT_OK = False
if not INFERENCE_OK:
    raise RuntimeError('Сначала выполните инференс текущего запуска без ошибок.')
missing = [str(p) for p in PREDICTIONS.values() if not p.is_file()]
if missing:
    raise RuntimeError('Метод не создал все результаты: ' + ', '.join(missing))

def png_size(path):
    with Path(path).open('rb') as f:
        header = f.read(24)
    if header[:8] != b'\x89PNG\r\n\x1a\n':
        raise ValueError(f'Ожидался PNG: {path}')
    return list(struct.unpack('>II', header[16:24]))

for path in PREDICTIONS.values():
    png_size(path)
freeze = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'environment.txt').write_text(freeze)
RUN_CONFIG['command'] = COMMAND
RUN_CONFIG['status'] = 'completed'
RUN_CONFIG['completed_at_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))
EXPORTED = []
for sid, predicted in PREDICTIONS.items():
    out_dir = RESULTS_ROOT / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    output = out_dir / 'vosr.png'
    shutil.copy2(predicted, output)
    metadata = dict(RUN_CONFIG, sample_id=sid,
        input_size=png_size(INPUT_DIR / f'{sid}.png'), output_size=png_size(output),
        input_sha256=hashlib.sha256((INPUT_DIR / f'{sid}.png').read_bytes()).hexdigest(),
        output_sha256=hashlib.sha256(output.read_bytes()).hexdigest())
    info = out_dir / 'vosr.json'
    info.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
    EXPORTED.extend([output, info])
archive = WORK_ROOT / f'giraffe_vosr_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in EXPORTED:
        z.write(path, Path('results') / path.relative_to(RESULTS_ROOT))
    z.write(CONFIG_PATH, 'run.json')
    z.write(RUN_DIR / 'environment.txt', 'environment.txt')
EXPORT_OK = True
print(f'Сохранено {len(PREDICTIONS)} результатов: {archive}')


## 7. Просмотр

`hint.png` — референс для сравнения. Он не поступает в модель и может отличаться ракурсом или кадрированием.


In [ ]:
if not EXPORT_OK:
    raise RuntimeError('Сначала завершите сохранение результатов текущего запуска.')
from IPython.display import display, HTML, FileLink
import base64
import html
for sid, source in SAMPLES.items():
    items = [('Вход: degraded.png', source), ('Результат: vosr', RESULTS_ROOT / sid / 'vosr.png')]
    hint = source.parent / 'hint.png'
    if hint.is_file():
        items.append(('Референс (hint.png)', hint))
    panels = []
    for label, path in items:
        encoded = base64.b64encode(path.read_bytes()).decode('ascii')
        panels.append(f'<div style="max-width:32%;"><p>{html.escape(label)}</p><img style="width:100%;height:auto" src="data:image/png;base64,{encoded}"></div>')
    display(HTML('<h3>' + html.escape(sid) + '</h3><div style="display:flex;gap:12px;align-items:flex-start">' + ''.join(panels) + '</div>'))
# Относительная ссылка обслуживается Jupyter; в Kaggle архив также виден в Output.
display(FileLink(os.path.relpath(archive, Path.cwd())))
